# 04 CNN Model

This notebook builds and evaluates a simple 1D CNN for software effort prediction.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col]).values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_cnn = reshape_for_cnn(X_train)
X_test_cnn = reshape_for_cnn(X_test)

print("Using dataset:", dataset_path.name)
print("CNN input shape:", X_train_cnn.shape)

Using dataset: desharnais_processed.csv
CNN input shape: (64, 12, 1)


In [2]:
model = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
 )

model, history = train_cnn_model(
    model,
    X_train_cnn, y_train,
    X_test_cnn, y_test,
    epochs=50,
    batch_size=32,
    verbose=0,
 )

print("Final validation MAE:", history["val_mean_absolute_error"][-1])

Final validation MAE: 4440.18310546875


In [3]:
cnn_preds = model.predict(X_test_cnn).ravel()
cnn_metrics = evaluate_predictions("cnn", y_test, cnn_preds)
cnn_metrics

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


,model,mae,rmse
0,cnn,4440.183594,5690.166254


In [4]:
cnn_metrics_path = save_metrics(
    cnn_metrics,
    file_name="cnn_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved CNN metrics to:", cnn_metrics_path)

Saved CNN metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\cnn_metrics.csv
